# 03 — Gold E-commerce Analytics

**Purpose:** Incrementally maintain business-ready analytical tables from trusted Silver events.

**Gold tables and grain:**

| Table | Grain |
|---|---|
| `daily_funnel_metrics` | one row per `event_date` |
| `daily_revenue_metrics` | one row per `event_date` |
| `product_daily_performance` | one row per `event_date` + `product_id` |

Gold uses `foreachBatch` + Delta `MERGE` rather than blind append so existing aggregate keys can be updated when late data affects a date that already exists.


In [ ]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F

SILVER_TABLE = "ecommerce_lakehouse.silver.transactions_clean"

GOLD_FUNNEL_TABLE = "ecommerce_lakehouse.gold.daily_funnel_metrics"
GOLD_REVENUE_TABLE = "ecommerce_lakehouse.gold.daily_revenue_metrics"
GOLD_PRODUCT_TABLE = "ecommerce_lakehouse.gold.product_daily_performance"

GOLD_CHECKPOINT = (
    "/Volumes/ecommerce_lakehouse/raw/pipeline_metadata/"
    "gold_transactions_checkpoint"
)


## Read new Silver commits

The Gold job consumes Silver as a streaming Delta source. Its checkpoint tracks which Silver commits have already been processed.


In [ ]:
silver_stream = (
    spark.readStream
        .table(SILVER_TABLE)
)


## Delta MERGE helper

New Gold keys are inserted; existing keys are updated. If a Gold table does not yet exist, the first batch creates it.


In [ ]:
def merge_into_gold(source_df, target_table, merge_condition):
    if not spark.catalog.tableExists(target_table):
        (
            source_df.write
                .format("delta")
                .saveAsTable(target_table)
        )
        return

    target = DeltaTable.forName(spark, target_table)

    (
        target.alias("target")
            .merge(
                source_df.alias("source"),
                merge_condition
            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
    )


## Recompute only affected dates

A micro-batch tells the pipeline which dates changed. The function then re-reads the complete Silver data for those dates and recomputes the Gold aggregates before merging them.

This is important for exact metrics such as `countDistinct()`: distinct counts from separate micro-batches cannot safely be added because the same user/session can appear in more than one batch.

The small set of affected dates is broadcast during the join. Manual DataFrame `cache()` / `persist()` is intentionally avoided because this project runs on Databricks serverless compute, where those APIs are unsupported.


In [ ]:
def process_gold_batch(microbatch_df, batch_id):
    if microbatch_df.isEmpty():
        return

    affected_dates_df = (
        microbatch_df
            .select("event_date")
            .distinct()
    )

    affected_silver_df = (
        spark.table(SILVER_TABLE)
            .join(
                F.broadcast(affected_dates_df),
                on="event_date",
                how="inner"
            )
    )

    # Gold 1: daily behavioral/event metrics
    daily_funnel_df = (
        affected_silver_df
            .groupBy("event_date")
            .agg(
                F.count("*").alias("total_events"),
                F.sum(
                    F.when(F.col("event_type") == "view", 1).otherwise(0)
                ).alias("views"),
                F.sum(
                    F.when(F.col("event_type") == "cart", 1).otherwise(0)
                ).alias("carts"),
                F.sum(
                    F.when(
                        F.col("event_type") == "remove_from_cart", 1
                    ).otherwise(0)
                ).alias("cart_removals"),
                F.sum(
                    F.when(F.col("event_type") == "purchase", 1).otherwise(0)
                ).alias("purchases"),
                F.countDistinct("user_id").alias("unique_users"),
                F.countDistinct("user_session").alias("unique_sessions"),
            )
            .withColumn(
                "view_to_cart_rate",
                F.when(
                    F.col("views") > 0,
                    F.round(F.col("carts") / F.col("views"), 4)
                )
            )
            .withColumn(
                "cart_to_purchase_rate",
                F.when(
                    F.col("carts") > 0,
                    F.round(F.col("purchases") / F.col("carts"), 4)
                )
            )
            .withColumn(
                "view_to_purchase_rate",
                F.when(
                    F.col("views") > 0,
                    F.round(F.col("purchases") / F.col("views"), 4)
                )
            )
    )

    merge_into_gold(
        daily_funnel_df,
        GOLD_FUNNEL_TABLE,
        "target.event_date = source.event_date"
    )

    # Gold 2: trusted purchase/revenue metrics
    valid_purchases_df = (
        affected_silver_df
            .filter(
                (F.col("event_type") == "purchase")
                & (F.col("_price_quality") == "VALID")
            )
    )

    daily_revenue_df = (
        valid_purchases_df
            .groupBy("event_date")
            .agg(
                F.count("*").alias("purchase_events"),
                F.round(
                    F.sum("price"),
                    2
                ).alias("purchased_item_revenue"),
                F.round(
                    F.avg("price"),
                    2
                ).alias("avg_purchased_item_price"),
                F.countDistinct("user_id").alias("unique_buyers"),
            )
    )

    merge_into_gold(
        daily_revenue_df,
        GOLD_REVENUE_TABLE,
        "target.event_date = source.event_date"
    )

    # Gold 3: product-level daily performance
    product_daily_df = (
        affected_silver_df
            .groupBy("event_date", "product_id")
            .agg(
                F.sum(
                    F.when(F.col("event_type") == "view", 1).otherwise(0)
                ).alias("views"),
                F.sum(
                    F.when(F.col("event_type") == "cart", 1).otherwise(0)
                ).alias("carts"),
                F.sum(
                    F.when(F.col("event_type") == "purchase", 1).otherwise(0)
                ).alias("purchases"),
                F.round(
                    F.sum(
                        F.when(
                            (F.col("event_type") == "purchase")
                            & (F.col("_price_quality") == "VALID"),
                            F.col("price")
                        ).otherwise(0.0)
                    ),
                    2
                ).alias("purchased_item_revenue"),
                F.countDistinct("user_id").alias("unique_users"),
            )
    )

    merge_into_gold(
        product_daily_df,
        GOLD_PRODUCT_TABLE,
        (
            "target.event_date = source.event_date "
            "AND target.product_id = source.product_id"
        )
    )


### Metric semantics

The columns named `*_rate` are ratios of independent event counts. They are **not strict user/session conversion probabilities**, so a value above `1` can be valid when purchase events exceed cart events for a date.


## Run the incremental Gold refresh

`availableNow=True` processes all currently unprocessed Silver commits and then stops, which makes the notebook suitable for scheduled Lakeflow Job execution.


In [ ]:
gold_query = (
    silver_stream.writeStream
        .foreachBatch(process_gold_batch)
        .option(
            "checkpointLocation",
            GOLD_CHECKPOINT
        )
        .trigger(availableNow=True)
        .start()
)

gold_query.awaitTermination()
